# 06 — Repair during evolutionary search

Test whether applying the existing consistency repair **before evolutionary fitness evaluation** finds more valid evasions than repairing final outputs.

**CPU runtime → Runtime → Run all. No previous ZIP upload is needed.** Keep the defaults. Download and share the new results ZIP at the end.

This is a larger development experiment: a fresh original CAA baseline and two in-search variants, for two models and three attack seeds (**18 CAA runs**). It takes longer than notebook 05. Existing files are preserved.


## 1. Fixed settings and a new workspace

The baseline and variants run in the same environment. Strict checks require identical pre-MOEVA candidates and matching core model-evaluation counts. Extra repair time and postprocessing are reported separately. No new IDS is trained.


In [ ]:
import datetime as dt
import hashlib
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import uuid

DEVELOPMENT_RECORDS = 32 #@param {type:"integer"}
SELECTION_SEED = 0 #@param {type:"integer"}
ATTACK_SEEDS = "0,1,2" #@param {type:"string"}
ATTACK_STEPS = 10 #@param {type:"integer"}
GENERATIONS = 100 #@param {type:"integer"}
OFFSPRING = 50 #@param {type:"integer"}
EPSILON = 0.5 #@param {type:"number"}
SEEDS = [int(value.strip()) for value in ATTACK_SEEDS.split(",") if value.strip()]
if not SEEDS or len(set(SEEDS)) != len(SEEDS) or min(SEEDS + [SELECTION_SEED]) < 0:
    raise ValueError("Enter distinct nonnegative attack seeds, separated by commas.")
if min(DEVELOPMENT_RECORDS, ATTACK_STEPS, GENERATIONS, OFFSPRING) < 1 or EPSILON <= 0:
    raise ValueError("Counts and epsilon must be positive.")
SAVE_TO_DRIVE = False #@param {type:"boolean"}
DOWNLOAD_RESULTS = True #@param {type:"boolean"}
REPO_URL = "https://github.com/Abdulstar/MyPhDThesis.git"
ROOT = Path(os.environ.get("PHD_WORKSPACE", "/content/MyPhDThesis_RepairDuringSearch")).resolve()
HOST_PYTHON, HOST_VERSION = sys.executable, sys.version
if not ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(ROOT)], check=True)
elif not (ROOT / ".git").is_dir():
    raise RuntimeError(f"{ROOT} is not a git checkout. Choose a new workspace.")
for required in ["scripts/bootstrap.py", "baseline/run_inloop_repair.py", "notebooks/06_Repair_During_Search.ipynb"]:
    if not (ROOT / required).is_file():
        raise RuntimeError(f"Missing {required}. Use a fresh clone of the updated repository.")

# Existing checkouts are preserved; no pull/reset/checkout is performed.
def unique_name(prefix):
    return prefix + "_" + dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]

SESSION = ROOT / "runs" / unique_name("repair_during_search")
SESSION.mkdir(parents=True, exist_ok=False)
source_paths = [p for folder in ["baseline", "scripts", "tests"] for p in (ROOT / folder).glob("*") if p.is_file()]
source_paths += [ROOT / "notebooks/06_Repair_During_Search.ipynb", ROOT / "docs/INLOOP_REPAIR_GUIDE.md"]
context = {
    "project_repository": REPO_URL,
    "project_commit": subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip(),
    "project_status": subprocess.check_output(["git", "-C", str(ROOT), "status", "--porcelain"], text=True).splitlines(),
    "session_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
    "notebook_python": HOST_VERSION, "notebook_executable": HOST_PYTHON,
    "source_sha256": {str(p.relative_to(ROOT)): hashlib.sha256(p.read_bytes()).hexdigest() for p in source_paths},
    "experiment": "repair_during_evolutionary_search", "evaluation_device": "cpu",
    "development_records": DEVELOPMENT_RECORDS, "selection_seed": SELECTION_SEED,
    "attack_seeds": SEEDS, "steps": ATTACK_STEPS, "generations": GENERATIONS,
    "offspring": OFFSPRING, "epsilon_scaled_l2": EPSILON,
}
(SESSION / "session_context.json").write_text(json.dumps(context, indent=2) + "\n")
for p in source_paths:
    target = SESSION / "source_snapshot" / p.relative_to(ROOT)
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(p, target)

def run_command(command, log_name):
    log_path = SESSION / log_name
    if log_path.exists():
        log_path = SESSION / (unique_name(Path(log_name).stem) + ".log")
    with log_path.open("w") as log:
        log.write("Command: " + json.dumps([str(x) for x in command]) + "\n")
        log.flush()
        process = subprocess.Popen([str(x) for x in command], cwd=ROOT, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
        code = process.wait()
    if code:
        raise RuntimeError(f"Command failed ({code}); see {log_path}. Export partial logs with the last cell.")

print("Project commit:", context["project_commit"])
print("New session:", SESSION)


## 2. Prepare the pinned CPU environment

The experiment uses isolated Python 3.8.20, NumPy 1.23.5 and CPU PyTorch. Your notebook kernel stays unchanged. Setup runs the project tests, including the repair-before-fitness checks.


In [ ]:
run_command([HOST_PYTHON, ROOT / "scripts/bootstrap.py"], "setup.log")
PYTHON = ROOT / ".baseline-env/bin/python"
assert sys.executable == HOST_PYTHON and sys.version == HOST_VERSION
setup_copy = SESSION / unique_name("setup_report")
shutil.copy2(ROOT / ".cache/setup_report.json", setup_copy.with_suffix(".json"))
print("Pinned audit interpreter:", PYTHON)


## 3. Download and verify the pinned assets

The fresh setup downloads approximately 348 MB of CTU feature data and the small checkpoints. These are aggregate records, not packets. All asset hashes are verified.

The compatibility scaler uses all feature rows to match frozen checkpoints. Only validation records are selected and scored. Later model training requires train-only preprocessing and independent final evaluation.


In [ ]:
run_command([PYTHON, ROOT / "baseline/download_assets.py", "--assets", ROOT / "assets"], "download.log")
manifest_copy = SESSION / (unique_name("download_manifest") + ".json")
shutil.copy2(ROOT / "assets/download_manifest.json", manifest_copy)


## 4. Run the fresh baseline and in-search variants

The first command creates the three notebook-05 comparisons: original CAA, budget-only end repair, and relationship + budget end repair.

The second command runs budget-only and relationship + budget repair inside MOEVA, after mutation and before fitness evaluation. It writes repaired values into the real evolutionary population. Original sampling, operators, objectives, generations and population sizes are preserved.

Progress appears after each model/seed/method and periodically during MOEVA. Keep the session connected. Repeated and clean candidate evaluations still count against the same generation budget.


In [ ]:
BASELINE_RUN = SESSION / "baseline"
RUN = SESSION / "inloop"
run_command([PYTHON, ROOT / "baseline/run_development_repair.py",
             "--repo", ROOT / "vendor/tabularbench", "--assets", ROOT / "assets",
             "--out", BASELINE_RUN, "--n-dev", DEVELOPMENT_RECORDS,
             "--selection-seed", SELECTION_SEED, "--seeds", *SEEDS,
             "--steps", ATTACK_STEPS, "--generations", GENERATIONS,
             "--offspring", OFFSPRING, "--eps", EPSILON], "baseline.log")
run_command([PYTHON, ROOT / "baseline/run_inloop_repair.py",
             "--repo", ROOT / "vendor/tabularbench", "--assets", ROOT / "assets",
             "--baseline-run", BASELINE_RUN, "--out", RUN], "inloop.log")
print("Completed paired experiment:", RUN)


## 5. Compare feasibility, evasion and cost

Coverage counts original records with a changed, valid, in-budget candidate in the saved final stage pools. It excludes clean fallbacks. New evasion uses initially detected records as its denominator.

For in-search methods, `newly_evaded` is the standalone result. `union_newly_evaded` also retains original CAA successes and requires both attack runs. The union is a separate, more expensive result. End-only methods already retain original successes.

Matching core model-evaluation counts does not mean matching runtime. Seed summaries describe repeated search on the same inputs, not independent population confidence intervals.


In [ ]:
import pandas as pd
from IPython.display import display

protocol = json.loads((RUN / "protocol.json").read_text())
results = json.loads((RUN / "results.json").read_text())
assert protocol["status"] == results["status"] == "completed"
assert not protocol["test_partition_evaluated"]
comparison = pd.read_csv(RUN / "comparison.csv")
summary = pd.read_csv(RUN / "seed_summary.csv")
assert comparison["generation_forward_counts_match_original"].all()
print("Completed comparison rows:", len(comparison))
print("Original and modified core attack forward counts match.")
display(comparison[["model", "seed", "method", "clean_detected", "covered_records",
                    "newly_evaded", "union_newly_evaded", "generation_forward_rows",
                    "generation_wall_seconds", "additional_postprocess_forward_rows"]])
display(summary)
print("Standalone and union evasion have different computation costs. Coverage is not IDS accuracy.")
print("Full per-generation audits and record-level evidence:", RUN)


## 6. Download the full evidence

The new ZIP includes both the fresh baseline and in-search comparison, per-generation audit records, raw final-stage arrays, validity masks, final representatives, model counters, source snapshots and logs. The dataset and environment are excluded.

**Share this ZIP for review.** If a preceding cell fails, run this export cell separately to preserve the partial results and logs.


In [ ]:
EXPORTS = ROOT / "exports"
EXPORTS.mkdir(exist_ok=True)
export_base = EXPORTS / unique_name(SESSION.name + "_results")
archive = Path(shutil.make_archive(str(export_base), "zip", root_dir=SESSION.parent, base_dir=SESSION.name))
print("Results ZIP:", archive)
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    destination = Path("/content/drive/MyDrive/MyPhDThesis/results")
    destination.mkdir(parents=True, exist_ok=True)
    target = destination / archive.name
    if target.exists():
        raise FileExistsError(f"Preserving existing archive: {target}")
    shutil.copy2(archive, target)
    print("Copied results to Google Drive:", target)
if DOWNLOAD_RESULTS:
    try:
        from google.colab import files
    except ImportError:
        from IPython.display import FileLink
        display(FileLink(str(archive)))
    else:
        files.download(str(archive))


## Interpret the result

- Additional valid evasions at a matched core query budget would support investigating repair during search; inspect added CPU work too.
- Improved feasibility without additional evasion is a feasibility result. It does not demonstrate improved IDS accuracy.
- Clean or repeated offspring, or fewer standalone evasions, indicate that conservative rollback may limit exploration.

This is a baseline research experiment. No novelty, optimality, live-traffic realizability or general robustness is established by a successful run. Preserve negative results and freeze independent evaluation before assessing the intended IDS defense contribution.
